Termolab python


1.) Beregn massestrømmen av kuldemedia $\dot{m}, [\frac{kg}{s}]$, for hver målingsserie dere har registrert ved å bruke energibalansen til et åpent system:

$$
\dot{W}_{komp} = \dot{m}(h_{inn}-h_{ut})
$$
 
som gir oss
$$
\frac{\dot{W}_{komp}}{(h_{inn}-h_{ut})} = \dot{m}
$$
hvor 
$$
h_{ut} = h_{inn} + \frac{h_{2,is}-h_{inn}}{\eta_{is}}
$$
og 
$$
\eta_{is} = 0.0017\pi^3 - 0.0286\pi^2 + 0.1533\pi + 0.1642
$$
$$
\pi = \frac{p_{kondenser}}{p_{fordamper}}
$$

In [48]:
import pandas as pd
import numpy as np

# m trenger: W, h_inn, h_ut, 
# h_ut trenger: h_inn, h_2,is, eta_is
# eta_is trenger: p_kond, p_ford
# p_kond, p_ford trenger: t_kond, t_ford
# h_2,is trenger: 

df = pd.read_excel("merged log termo.xlsx", sheet_name="log")  

df.columns = ["time", "T1", "T2", "T3", "T4", "Thot", "Tcold", "W_komp", "Tkond", "Tford"]   
            #hh:mm:ss, degC,, degC, degC, degC, degC, degC,      W,       degC,    degC 
# finner W
W_komp = df["W_komp"].to_numpy()


In [49]:
from CoolProp.CoolProp import PropsSI
#finner entalpier


fluid = "R134a"
kelvin = 273.15
manuel_tid= ["12:24:01", "12:25:59", "12:28:08", "12:30:06", "12:32:04"]

log_date = df["time"].dt.date.iloc[0]

start_time, end_time = pd.to_datetime([
    f"{log_date} {manuel_tid[0]}",
    f"{log_date} {manuel_tid[-1]}"])

yippie = df.loc[df["time"].between(start_time, end_time)].copy()

def entalpi1(data):

    """
    input: data -> dataset
    1. lager kopi av dataset
    2. finner fordampnings- og konsenseringstrykk
    3. bruker trykkene for å finne den isentropiske virkningsgraden
    4. finenr h1, h2, h2is ved hjelp av dataene i forrige steg
    5. returnerer dataset med 
    """

    data = data.copy()
    T_suc  = data["T1"].to_numpy() + kelvin   # [K]
    p_ford = PropsSI("P", "T", data["Tford"].to_numpy() + kelvin, "Q", 1, fluid)   # [Pa]
    p_kond = PropsSI("P", "T", data["Tkond"].to_numpy() + kelvin, "Q", 1, fluid)   # [Pa]

    data["superheat_K"] = data["T1"] - data["Tford"]            
    data["pi"] = p_kond / p_ford
    pi = data["pi"]
    data["eta_is"] = 0.0017*pi**3 - 0.0286*pi**2 + 0.1533*pi + 0.1642

    h1   = PropsSI("H", "T", T_suc, "P", p_ford, fluid)       # [J/kg]
    s1   = PropsSI("S", "T", T_suc, "P", p_ford, fluid)       # [J/kgK]
    h2is = PropsSI("H", "P", p_kond, "S", s1, fluid)          # isentropisk 
    h2   = h1 + (h2is - h1) / data["eta_is"].to_numpy()       # ekte

    data["h1"], data["h2is"], data["h2"] = h1, h2is, h2       # [J/kg]
    return data

res = entalpi1(yippie)
res_manual = res[res["time"].dt.strftime("%H:%M:%S").isin(manuel_tid)]
print(res_manual[["time", "eta_is", "h1", "h2", "h2is"]])

                  time    eta_is             h1             h2           h2is
27 2026-10-06 12:24:01  0.396674  423448.046318  477115.550893  444736.562076
38 2026-10-06 12:25:59  0.411898  427709.421687  489917.418805  453332.749961
50 2026-10-06 12:28:08  0.421804  432295.215675  502613.469525  461955.740381
61 2026-10-06 12:30:06  0.428598  435674.298241  515587.507109  469924.934773
72 2026-10-06 12:32:04  0.429792  438680.891418  524540.959219  475582.827625


In [ ]:
res["m_dot"] = res["W_komp"] / (res["h2"] - res["h1"])     # [kg/s]
res["m_dot_e3"] = res["m_dot"] * 1e3                        # [1e-3 kg/s]

date = df["time"].dt.date.iloc[0]
manuel_tid = pd.to_datetime([f"{date} {t}" for t in
    ["12:24:01", "12:25:59", "12:28:08", "12:30:06", "12:32:04"]])
res_manual = res[res["time"].isin(manuel_tid)]

print(res_manual["m_dot"])

27    0.002739
38    0.002395
50    0.002147
61    0.001915
72    0.001817
Name: m_dot, dtype: float64
